## Importação de Bibliotecas

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Manipulação dos dados 
import pandas as pd
import numpy as np

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Funções customizadas
from configs.paths import *
from configs.function_basic import *

# Avisos
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('✅ Bibliotecas carregadas com sucesso')

## Carregamento dos Dados

In [ ]:
# Carregar dados
abt00 = pd.read_parquet(RAW_DIR / 'base_tabelao.parquet')

print(f'✅ Dados carregados')

## Análise Inicial dos Dados

In [ ]:
# Informações básicas
print('imformações básicas'.upper())
print('=' * 30)

basic_information(abt00)

In [ ]:
# Informações dos tipos de dados
print('tipo dos dados'.upper())
print('=' * 30)

data_type(abt00)

 Ajuste de Tipos de Dados

Para garantir consistência analítica e evitar problemas ao longo do pipeline, será necessário ajustar o tipo de dados de algumas colunas:

- **DATADENASCIMENTO**: conversão para o tipo `datetime`, permitindo operações temporais, validações e cálculos de idade.

Esse ajuste reduz risco operacional e aumenta a robustez do dataset para etapas posteriores.

In [ ]:
# Visualizando os dados 
print('Visualizando os dados'.upper())
print('=' * 30)
show_dataframe_samples(abt00)

### Visão Geral do Dataset

O dataset representa uma base de clientes associada a uma **safra específica**, consolidando informações cadastrais, comportamentais, scores externos e variáveis derivadas para análise de risco e performance.

### Estrutura das Variáveis

- **Identificação e Safra**
  - `SAFRA`: referência temporal do dataset (formato AAAAMM).
  - `NUM_CPF`: identificador único do cliente (anonimizado).

- **Produto e Status**
  - `PROD`: tipo de produto contratado.
  - `STATUSRF`: status regulatório do cliente.
  - `FLAG_INSTALACAO`, `FLAG_INSTALACAO_cadastrais`: indicadores de instalação do serviço.

- **Indicadores de Risco e Performance**
  - `FPD`, `FPD_bureau`, `FPD_telco`: flags de First Payment Default por origem.
  - `flag_mig2`, `flag_mig2_bureau`, `flag_mig2_telco`: indicadores de migração de risco.
  - `SCORE_01`, `SCORE_02`: scores preditivos utilizados no processo decisório.

- **Variáveis Derivadas**
  - `var_02` a `var_93`: conjunto de variáveis numéricas derivadas, utilizadas para modelagem estatística e machine learning.  
    Incluem métricas normalizadas, percentuais, contagens e indicadores comportamentais.

- **Dados Cadastrais**
  - `DATADENASCIMENTO`: data de nascimento do cliente.
  - `CEP_3_digitos`: agregação geográfica do CEP (3 primeiros dígitos).

### Observações Importantes

- O dataset apresenta **valores nulos** em algumas variáveis derivadas, exigindo tratamento prévio.
- Existem variáveis com **alta repetição de valores constantes**, possivelmente flags ou indicadores técnicos.
- Recomenda-se validação de tipos (`datetime`, `int`, `float`) antes de qualquer etapa de modelagem.
- A granularidade das variáveis sugere uso principal em **modelos de crédito / risco**.

## Análise da Variável Target (FPD)

In [ ]:
# Análise do target
analyze_target(abt00, target_col='FPD')

In [ ]:
# Estatísticas do target
total = len(abt00)
bons = (abt00['FPD'] == 0).sum()
maus = (abt00['FPD'] == 1).sum()

print(f'\n📊 DISTRIBUIÇÃO DO TARGET (FPD):')
print(f'   Total de registros: {total:,}')
print(f'   Bons pagadores (0): {bons:,} ({bons/total*100:.2f}%)')
print(f'   Maus pagadores (1): {maus:,} ({maus/total*100:.2f}%)')
print(f'   Taxa de inadimplência: {maus/total*100:.2f}%')

# Verificar balanceamento
ratio = bons / maus
print(f'\n⚖️ Desbalanceamento: {ratio:.2f}:1 (bons:maus)')
if ratio > 2:
    print('   ⚠️ Dataset desbalanceado - considerar técnicas de balanceamento')

## Análise de Valores Faltantes

In [ ]:
# Análise de missing values
missing_df = analyze_missing_values(abt00, plot=False)

# Visualizar top 30 variáveis com missing
if not missing_df.empty:
    top_missing = missing_df.head(20)
    
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(top_missing)), top_missing['Percentage'])
    plt.yticks(range(len(top_missing)), top_missing['Variable'])
    plt.xlabel('Percentual de Valores Faltantes (%)')
    plt.title('Top 30 Variáveis com Valores Faltantes')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## Análise de Duplicatas

In [ ]:
# Verificação de duplicatas
print('análise de duplicatas'.upper())
print('=' * 30)

check_duplicates(abt00)

## Tabela Resumo do Dataset

In [ ]:
# Tabela de informações
metadados = dataset_info_table(abt00)
metadados

In [ ]:
# Salvar tabela
metadados.to_csv(ARTIFACT_DIR / 'metadados.csv', index=False)
print(f'\n✅ Tabela salva em: {ARTIFACT_DIR / "metadados.csv"}')

## Análise de Variáveis Numéricas

In [ ]:
# Selecionar variáveis numéricas (exceto target)
numerical_cols = abt00.select_dtypes(include=[np.number]).columns.tolist()

for col in ['FPD', 'FPD_bureau', 'FPD_telco']:
    if col in numerical_cols:
        numerical_cols.remove(col)

print(f'\n📊 Total de variáveis numéricas: {len(numerical_cols)}')
print(f'\nExemplos: {numerical_cols[:10]}')

In [ ]:
# estatísticas descritivas das principais features numéricas
main_vars = ['SCORE_01', 'SCORE_02']

existing_vars = [v for v in main_vars if v in abt00.columns]

if existing_vars:
    print('\n📊 ESTATÍSTICAS DESCRITIVAS - FEATURES PRINCIPAIS:')
    display(abt00[['SCORE_01', 'SCORE_02']].describe().round(2))

- Os scores apresentam comportamentos distintos. O SCORE_02 possui média mais elevada e maior dispersão, indicando maior poder de discriminação entre perfis de risco. Já o SCORE_01 é mais concentrado, sugerindo uma régua mais conservadora e estável. A presença de valores mínimos iguais a zero no SCORE_01 indica a necessidade de validação de regra ou tratamento específico.  
SCORE_02 separa melhor, SCORE_01 dá estabilidade.


In [ ]:
# Distribuições das principais variáveis numéricas
if existing_vars:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes = axes.flatten()
    
    for i, var in enumerate(existing_vars):
        if i < 4:
            abt00[var].hist(bins=50, ax=axes[i], edgecolor='black')
            axes[i].set_title(f'Distribuição: {var}')
            axes[i].set_xlabel(var)
            axes[i].set_ylabel('Frequência')
    
    plt.tight_layout()
    plt.show()

## Análise de Variáveis Categóricas

In [ ]:
# Análise de variáveis categóricas
categorical_cols = abt00.select_dtypes(include=['object']).columns.tolist()

print(f'\n📊 Total de variáveis categóricas: {len(categorical_cols)}')
print(f'\nVariáveis: {categorical_cols}')

In [ ]:
# Analisar principais categóricas
if categorical_cols:
    analyze_categorical_features(abt00[categorical_cols], max_unique=20)

- STATUSRF preservar a categoria REGULAR e agrupar os demais status em OUTROS, garantindo simplificação da variável, maior robustez do modelo e melhor capacidade de generalização.
- CEP_3_DIGITOS Foi agrupada em regiões macro para reduzir complexidade, evitar overfitting e melhorar a estabilidade do modelo.

## Grupo controle

In [ ]:
# resumo do grupo controle (CPF 6º e 7º dígitos = ZZ ou ZX)
mask_grupo_controle = abt00['NUM_CPF'].astype(str).str[5:7].isin(['ZZ', 'ZX'])

qt_grupo_controle = mask_grupo_controle.sum()
qt_total = len(abt00)
pct = qt_grupo_controle / qt_total * 100

print("📌 GRUPO CONTROLE – CPF (6º e 7º dígitos = ZZ ou ZX)")
print(f"• Total de registros        : {qt_total:,}")
print(f"• Registros grupo controle  : {qt_grupo_controle:,}")
print(f"• Representatividade        : {pct:.2f}%")


> O grupo controle representa 4,6% da base e é composto por clientes aprovados fora das regras tradicionais de score. Para evitar viés no aprendizado e avaliar o impacto desse comportamento excepcional, será conduzida uma experiência de modelagem excluindo esse grupo.
>
> Posteriormente, o grupo será explicitamente sinalizado por meio de uma flag, permitindo isolar esse efeito e garantir que a régua principal de risco permaneça coerente e não contaminada  por decisões fora do fluxo padrão.

## Correlação com o Target

In [ ]:
# calcula correlação real das variáveis com o target FPD
correlations = (abt00[numerical_cols + ['FPD']].corr()['FPD'].drop('FPD').sort_values(ascending=False))

In [ ]:
# separa correlações positivas e negativas
corr_pos = correlations[correlations > 0]
corr_neg = correlations[correlations < 0]

In [ ]:
print('\n📈 TOP 10 CORRELAÇÕES POSITIVAS COM FPD:')
print(corr_pos.head(10))

print('\n📉 TOP 10 CORRELAÇÕES NEGATIVAS COM FPD:')
print(corr_neg.tail(10))

A análise de correlação evidencia dois comportamentos distintos. As variáveis com correlação positiva apresentam impacto incremental sobre o risco de FPD,  
enquanto os scores possuem correlação negativa mais intensa, indicando efeito protetivo consistente. Esse resultado reforça o papel central dos scores na modelagem, com as demais variáveis atuando como complementares no processo decisório.


In [ ]:
# plota top correlações negativas
top_neg = corr_neg.sort_values().head(10)

plt.figure(figsize=(8, 6))
plt.barh(top_neg.index, top_neg.values)
plt.xlabel('Correlação com FPD')
plt.title('Top 10 Variáveis com Correlação Negativa com FPD')
plt.axvline(x=0, linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

### Correlação das Variáveis com o FPD

As variáveis com maior correlação absoluta com o target **FPD** apresentam, em sua maioria, **associações fracas a moderadas**, com destaque para os scores externos.

- **SCORE_02 (ρ ≈ 0,29)** e **SCORE_01 (ρ ≈ 0,21)** são as variáveis mais informativas, reforçando seu papel como principais indicadores de risco.
- As demais variáveis (`var_XX`) possuem correlações individuais inferiores a 0,10, sugerindo **baixo poder explicativo isolado**.
- A ausência de correlações elevadas indica que o risco não é explicado por uma única variável, mas sim pela **combinação de múltiplos sinais**.

### Implicações Analíticas

- Scores devem ser tratados como **features-chave** no processo de modelagem.
- Variáveis derivadas contribuem de forma incremental e tendem a performar melhor em **modelos multivariados**.
- Não há evidência de vazamento de informação, dado que nenhuma variável apresenta correlação excessivamente alta com o target.

Esse cenário reforça a necessidade de uso de **modelos estatísticos ou de machine learning**, em vez de regras univariadas.


## Análise por SAFRA

In [ ]:
# Calcule a média do FPD e o volume por AAAAMM
resultado = abt00.groupby('SAFRA').agg({'FPD': 'mean', 'SAFRA': 'count'}).rename(columns={'SAFRA': 'Volume'}).reset_index()
resultado.columns = ['Safra (AAAAMM)', 'Taxa_de_Evento', 'Volume']

# Exiba a tabela
resultado

In [ ]:
# Gráfico com barras para o volume e linha para a taxa de evento por safra
fig, ax1 = plt.subplots(figsize=(12, 6))

color = 'lightblue'  # Azul mais escuro
ax1.bar(resultado['Safra'], resultado['Volume'], color=color, label='Volume')
ax1.set_xlabel('Safra')
ax1.set_ylabel('Volume', color='black')
ax1.tick_params(axis='y', labelcolor='black')

ax2 = ax1.twinx()  # Cria um segundo eixo y
color = 'hotpink'  # Rosa sólido
ax2.plot(resultado['Safra'], resultado['Taxa_de_Evento'] * 100, marker='o', linestyle='-', color=color, label='Taxa de Evento (%)')
ax2.set_ylabel('Taxa de Evento (%)', color='black')
ax2.tick_params(axis='y', labelcolor='black')

for label in ax1.get_xticklabels() + ax1.get_yticklabels() + ax2.get_yticklabels():
    label.set_fontsize(12)  # Tamanho da fonte
    label.set_color('black')  # Cor sólida

plt.title('Volume e Taxa de Evento por Safra')
plt.legend(loc='upper left')
plt.show()

A análise por safra indica estabilidade no volume de clientes, com variação controlada ao longo do período. Observa-se um pico de inadimplência em 202411, seguido por redução gradual nas safras posteriores, sugerindo ajuste ou amadurecimento da política de concessão. O leve aumento em 202503 merece monitoramento, mas permanece abaixo do pior cenário observado.  
Cresceu rápido em 202411 e pagou o preço.  
Depois apertou o freio e a inadimplência respondeu.

## 12. Conclusões do Entendimento dos Dados

### Principais Conclusões

- **Volume e Complexidade:**  
  O dataset possui 1.272.095 registros e 108 variáveis, caracterizando uma base robusta e adequada para modelagem preditiva de risco.

- **Variável Target (FPD):**  
  A taxa de inadimplência é de aproximadamente 23,4%, com leve desbalanceamento da base (≈3,3:1). Esse cenário é adequado para modelagem, exigindo apenas atenção na escolha das métricas de avaliação.

- **Qualidade dos Dados:**  
  Observa-se presença relevante de valores faltantes em diversas variáveis, o que demanda estratégias de tratamento e imputação antes da modelagem.

- **Sinais Relevantes de Risco:**  
  Os scores de crédito (**SCORE_01** e **SCORE_02**) concentram o maior poder discriminatório individual, enquanto as demais variáveis atuam de forma complementar, reforçando a necessidade de abordagem multivariada.

- **Implicações Analíticas:**  
  Não há evidência de uma variável isolada capaz de explicar o FPD, o que reforça o uso de modelos estatísticos ou de machine learning em detrimento de regras univariadas.

- **Próximos Passos:**  
  - Tratamento de valores ausentes  
  - Feature engineering e seleção de variáveis  
  - Avaliação de métricas de separação (KS, Gini)  
  - Construção e validação de modelos preditivos
